In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
import sys

In [19]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent 
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.file_paths import TISSUE_DIMENSIONS
print("Available tissue dims:", TISSUE_DIMENSIONS.keys())

Available tissue dims: dict_keys(['cervix', 'brain', 'afmmm'])


In [20]:
# Add the parent directory to the path to access the utils module
sys.path.append('..')
from src.utils.file_paths import file_paths, TISSUE_DIMENSIONS

# Set display options for better visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 1. Load and Merge Brain Dataset Files

In [21]:
# Check if the brain raw path exists
if not file_paths.brain_raw_path.exists():
    print(f"Brain raw path does not exist: {file_paths.brain_raw_path}")
    sys.exit(1)
else:
    print(f"Brain raw path exists: {file_paths.brain_raw_path}")

# Create interim and combined directories
file_paths.brain_interim_path.mkdir(parents=True, exist_ok=True)
file_paths.combined_interim_path.mkdir(parents=True, exist_ok=True)

# Gather .npy files for brain
brain_files = list(file_paths.brain_raw_path.glob('**/*.npy'))
print(f"Found {len(brain_files)} .npy files in the brain raw directory.")

# Exclude specific brain samples
exclude_brain = {
    '2023-03-22_T_HORAO-91-BF_FR_S1_1_combined.npy',
    '2022-11-22_T_HORAO-67-BF_FR_S1_1_combined.npy',
    '2024-09-18_T_AUTOPSY-BF_FR_S4_1_combined.npy',
    '2023-05-26_T_HORAO-103-BF_FR_S1_1_combined.npy',
    '2023-05-16_T_HORAO-101-BF_FR_S1_1_combined.npy'
}
brain_files = [fp for fp in brain_files if fp.name not in exclude_brain]
print(f"{len(brain_files)} files remaining after excluding {len(exclude_brain)} brain samples.")


Brain raw path exists: /Users/chaechae/Desktop/EP_Code/pr_prediction/notebooks/../data/raw/brain
Found 146 .npy files in the brain raw directory.
141 files remaining after excluding 5 brain samples.


## 2. Analyze Sample File Structure

## 3. Merge and Process All Brain Files

In [14]:
def merge_brain_files(files, output_path):
    """
    Merge brain .npy files, save merged array and return info DataFrame.
    """
    dims = TISSUE_DIMENSIONS['brain']
    nrows, ncols = dims['num_rows'], dims['num_cols']

    arrays = []
    info = []
    for fp in tqdm(files, desc="Processing brain files"):
        try:
            arr = np.load(fp)
            if arr.shape[:2] == (nrows, ncols) and arr.shape[2] == 17:
                sid = fp.stem.replace('_combined', '')
                arrays.append(arr)
                info.append({
                    'sample_id': sid,
                    'file_path': str(fp),
                    'shape': arr.shape,
                    'mask_coverage': np.mean(arr[..., -1])
                })
            else:
                print(f"Skipping {fp.name}: unexpected shape {arr.shape}")
        except Exception as e:
            print(f"Error loading {fp.name}: {e}")

    if arrays:
        merged = np.stack(arrays, axis=0)
        np.save(output_path, merged)
        print(f"Saved merged brain array to {output_path} (shape: {merged.shape})")
        return merged, pd.DataFrame(info)
    else:
        print("No valid brain arrays to merge.")
        return None, pd.DataFrame(info)



In [15]:
# Merge and save brain
brain_merged_path = file_paths.brain_interim_path / 'brain_merged.npy'
brain_merged, brain_info = merge_brain_files(brain_files, brain_merged_path)
brain_info.to_csv(file_paths.brain_interim_path / 'brain_sample_info.csv', index=False)

Processing brain files:   0%|          | 0/141 [00:00<?, ?it/s]

Saved merged brain array to /Users/chaechae/Desktop/EP_Code/pr_prediction/notebooks/../data/interim/brain/brain_merged.npy (shape: (141, 388, 516, 17))


## 4. Prepare Data for ML Training

In [16]:
# Create X (features) and y (labels) datasets
if brain_merged is not None:
    # X: all samples, all pixels, first 16 channels (Mueller matrix)
    X_brain = brain_merged[..., :16]
    
    # y: all samples, all pixels, last channel (physical realizability mask)
    y_brain = brain_merged[..., -1]
    
    print(f"X_brain shape: {X_brain.shape}")
    print(f"y_brain shape: {y_brain.shape}")
    
    # Save the prepared X and y datasets
    X_brain_path = file_paths.brain_processed_path / 'X_brain.npy'
    y_brain_path = file_paths.brain_processed_path / 'y_brain.npy'
    
    # Create the directory if it doesn't exist
    file_paths.brain_processed_path.mkdir(parents=True, exist_ok=True)
    
    np.save(X_brain_path, X_brain)
    np.save(y_brain_path, y_brain)
    
    print(f"Saved X_brain to {X_brain_path}")
    print(f"Saved y_brain to {y_brain_path}")


X_brain shape: (141, 388, 516, 16)
y_brain shape: (141, 388, 516)
Saved X_brain to /Users/chaechae/Desktop/EP_Code/pr_prediction/notebooks/../data/processed/brain/X_brain.npy
Saved y_brain to /Users/chaechae/Desktop/EP_Code/pr_prediction/notebooks/../data/processed/brain/y_brain.npy


## Merge with other dataset

In [22]:
def merge_and_save_tissue(tissue_name, exclude_names):
    """
    Merge .npy files for a given tissue, exclude specified samples,
    and save merged features (X) and labels (y).
    """
    raw_path = getattr(file_paths, f"{tissue_name}_raw_path")
    processed_path = getattr(file_paths, f"{tissue_name}_processed_path")

    if not raw_path.exists():
        print(f"{tissue_name.capitalize()} raw path does not exist: {raw_path}")
        return
    processed_path.mkdir(parents=True, exist_ok=True)

    files = list(raw_path.glob('**/*.npy'))
    print(f"Found {len(files)} .npy files in the {tissue_name} raw directory.")

    files = [fp for fp in files if fp.name not in exclude_names]
    print(f"{len(files)} files remaining after excluding {len(exclude_names)} {tissue_name} samples.")

    dims = TISSUE_DIMENSIONS[tissue_name]
    nrows, ncols = dims['num_rows'], dims['num_cols']

    arrays, info = [], []
    for fp in tqdm(files, desc=f"Processing {tissue_name} files"):
        try:
            arr = np.load(fp)
            if arr.shape[:2] == (nrows, ncols) and arr.shape[2] == 17:
                sid = fp.stem.replace('_combined', '')
                arrays.append(arr)
                info.append({
                    'sample_id': sid,
                    'file_path': str(fp),
                    'shape': arr.shape,
                    'mask_coverage': np.mean(arr[..., -1])
                })
            else:
                print(f"Skipping {fp.name}: unexpected shape {arr.shape}")
        except Exception as e:
            print(f"Error loading {fp.name}: {e}")

    if arrays:
        merged = np.stack(arrays, axis=0)
        X = merged[..., :16]
        y = merged[..., -1]

        np.save(processed_path / 'merged_all_X.npy', X)
        np.save(processed_path / 'merged_all_y.npy', y)
        print(f"Saved {tissue_name} merged X (shape {X.shape}) and y (shape {y.shape}).")

        pd.DataFrame(info).to_csv(processed_path / f'{tissue_name}_sample_info.csv', index=False)
        print(f"Saved {tissue_name} sample info CSV.")
    else:
        print(f"No valid {tissue_name} arrays to merge.")

## Merge Cervix and AFMMM

In [23]:
exclude_cervix = {
    'Sample2_550_PR_combined.npy',
    'Sample18_550_PR_combined.npy',
    'Sample19_550_PR_combined.npy',
    'Sample23_550_PR_combined.npy',
    'Sample25_550_PR_combined.npy'
}
exclude_afmmm = {
    'AFMMM_sample_he9_PR_combined.npy',
    'AFMMM_sample_he14_PR_combined.npy',
    'AFMMM_sample_bg5_PR_combined.npy',
    'AFMMM_sample_bw6_PR_combined.npy',
    'AFMMM_sample_bg11_PR_combined.npy'
}

In [5]:
# Merge cervix and AFMMM
merge_and_save_tissue('cervix', exclude_cervix)

NameError: name 'file_paths' is not defined

In [24]:
merge_and_save_tissue('afmmm', exclude_afmmm)

Found 52 .npy files in the afmmm raw directory.
47 files remaining after excluding 5 afmmm samples.


Processing afmmm files:   0%|          | 0/47 [00:00<?, ?it/s]

Saved afmmm merged X (shape (47, 500, 500, 16)) and y (shape (47, 500, 500)).
Saved afmmm sample info CSV.


## Combine Datasets for ML Training

In [25]:
# Load processed X and y for each tissue
X_brain = np.load(file_paths.brain_processed_path / 'X_brain.npy')
y_brain = np.load(file_paths.brain_processed_path / 'y_brain.npy')
X_cervix = np.load(file_paths.cervix_processed_path / 'merged_all_X.npy')
y_cervix = np.load(file_paths.cervix_processed_path / 'merged_all_y.npy')
X_afmmm = np.load(file_paths.afmmm_processed_path / 'merged_all_X.npy')
y_afmmm = np.load(file_paths.afmmm_processed_path / 'merged_all_y.npy')


In [26]:
# Concatenate along the sample axis
X_all = np.concatenate([X_brain, X_cervix, X_afmmm], axis=0)
Y_all = np.concatenate([y_brain, y_cervix, y_afmmm], axis=0)

print(f"Combined X_all shape: {X_all.shape}")
print(f"Combined Y_all shape: {Y_all.shape}")


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 388 and the array at index 1 has size 600